# Tarea 2 – Extracción OCR (Corpus A: documentos)
- Fecha de extracción: 25/10/2026
- PaddleOCR 3.7.0, Python 3.11.17 (entorno conda `tarea2`)
- Alcance: solo Corpus A (documentos). El Corpus B (audio) no se incluyó en esta entrega.
- Asistente de IA usado: Claude (chat), capturas en `evidencias_ia/`
- Fuentes y licencias: ver `datos/fuentes.csv`

In [1]:
import os, hashlib, pandas as pd

carpeta = "datos/raw/documentos"
archivos = sorted(f for f in os.listdir(carpeta) if f.lower().endswith(".jpg"))
print(len(archivos), "archivos")

def tipo(n):
    return "impreso" if n <= 8 else ("foto" if n <= 16 else "manuscrito")

filas, hashes = [], {}
for f in archivos:
    n = int(f[4:7])
    ruta = f"{carpeta}/{f}"
    h = hashlib.sha256(open(ruta, "rb").read()).hexdigest()
    hashes.setdefault(h, []).append(f)
    filas.append({"id": f[:-4], "archivo": ruta, "tipo": tipo(n), "fuente": "",
                  "texto_ocr": "", "modelo_ocr": "", "texto_referencia": ""})

df = pd.DataFrame(filas)
print("Duplicados:", [v for v in hashes.values() if len(v) > 1])
print("MB mayores a 3:", [f for f in archivos if os.path.getsize(f"{carpeta}/{f}") > 3e6])
os.makedirs("datos", exist_ok=True)
df.to_csv("datos/documentos.csv", index=False)
pd.DataFrame({"id": df["id"], "modalidad": "documento", "fuente_url": "",
              "licencia": "", "acceso": ""}).to_csv("datos/fuentes.csv", index=False)
df.head()

25 archivos
Duplicados: []
MB mayores a 3: []


,id,archivo,tipo,fuente,texto_ocr,modelo_ocr,texto_referencia
0,doc_001,datos/raw/documentos/doc_001.jpg,impreso,,,,
1,doc_002,datos/raw/documentos/doc_002.jpg,impreso,,,,
2,doc_003,datos/raw/documentos/doc_003.jpg,impreso,,,,
3,doc_004,datos/raw/documentos/doc_004.jpg,impreso,,,,
4,doc_005,datos/raw/documentos/doc_005.jpg,impreso,,,,


In [1]:
import time
from paddleocr import PaddleOCR

ocr = PaddleOCR(lang="es",
                use_doc_orientation_classify=False,
                use_doc_unwarping=False,
                use_textline_orientation=False,
                enable_mkldnn=False)
t = time.time()
res = ocr.predict("datos/raw/documentos/doc_001.jpg")
print(round(time.time() - t, 1), "s")
print(res[0]["rec_texts"][:10])

d:\Python2026\envs\tarea2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Python2026\envs\tarea2\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\acer\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\acer\.paddlex\official_models\PP-OCRv6_medium_re

104.2 s
['1', 'CONSTITUCIÓN POLÍTICA DEL PERÚ', 'PREÁM BULO', 'EL CONGRESO CONSTITUYENTE DEMOCRÁTICO, INVOCANDO A DIOS', 'TODOPODEROSO, OBEDECIENDO EL MANDATO DEL PUEBLO PERUANO Y', 'RECORDANDO EL SACRIFICIO DE TODAS LAS GENERACIONES QUE NOS', 'HAN PRECEDIDO EN NUESTRA PATRIA, HA RESUELTO DAR LA SIGUIENTE', 'CONSTITUCION:', 'TÍTULOI', 'DE LA PERSONA Y DE LA SOCIEDAD']


In [2]:
import os, time, pandas as pd
from paddleocr import PaddleOCR

ocr = PaddleOCR(lang="es", use_doc_orientation_classify=False,
                use_doc_unwarping=False, use_textline_orientation=False,
                enable_mkldnn=False)

carpeta = "datos/raw/documentos"
filas = []
for f in sorted(os.listdir(carpeta)):
    if not f.lower().endswith(".jpg"):
        continue
    n = int(f[4:7])
    t = time.time()
    r = ocr.predict(f"{carpeta}/{f}")[0]
    texto = " ".join(r["rec_texts"])
    filas.append({"id": f[:-4], "archivo": f"{carpeta}/{f}",
                  "tipo": "impreso" if n <= 8 else ("foto" if n <= 16 else "manuscrito"),
                  "fuente": "", "texto_ocr": texto,
                  "modelo_ocr": "PaddleOCR 3.7.0 (lang=es)",
                  "texto_referencia": "", "segundos": round(time.time() - t, 1)})
    print(f, len(texto), "caracteres", round(time.time() - t, 1), "s")

df = pd.DataFrame(filas)
os.makedirs("datos", exist_ok=True)
df.drop(columns="segundos").to_csv("datos/documentos.csv", index=False)
df.to_csv("datos/raw/documentos/ocr_con_tiempos.csv", index=False)
print(df.shape)
df.head()

Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\acer\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\acer\.paddlex\official_models\PP-OCRv6_medium_rec`.


doc_001.jpg 2070 caracteres 103.0 s
doc_002.jpg 2492 caracteres 120.5 s
doc_003.jpg 2200 caracteres 104.8 s
doc_004.jpg 1874 caracteres 99.3 s
doc_005.jpg 2575 caracteres 124.4 s
doc_006.jpg 3384 caracteres 148.5 s
doc_007.jpg 3712 caracteres 177.2 s
doc_008.jpg 6393 caracteres 202.0 s
doc_009.jpg 3269 caracteres 71.0 s
doc_010.jpg 545 caracteres 131.7 s
doc_011.jpg 1138 caracteres 69.6 s
doc_012.jpg 1740 caracteres 77.7 s
doc_013.jpg 1857 caracteres 81.0 s
doc_014.jpg 490 caracteres 43.9 s
doc_015.jpg 672 caracteres 42.2 s
doc_016.jpg 848 caracteres 51.3 s
doc_017.jpg 2498 caracteres 171.0 s
doc_018.jpg 1103 caracteres 114.9 s
doc_019.jpg 983 caracteres 113.3 s
doc_020.jpg 1127 caracteres 112.6 s
doc_021.jpg 2306 caracteres 304.8 s
doc_022.jpg 1931 caracteres 139.8 s
doc_023.jpg 559 caracteres 106.0 s
doc_024.jpg 2548 caracteres 150.9 s
doc_025.jpg 2415 caracteres 156.7 s
(25, 8)


,id,archivo,tipo,fuente,texto_ocr,modelo_ocr,texto_referencia,segundos
0,doc_001,datos/raw/documentos/doc_001.jpg,impreso,,1 CONSTITUCIÓN POLÍTICA DEL PERÚ PREÁM BULO EL...,PaddleOCR 3.7.0 (lang=es),,103.0
1,doc_002,datos/raw/documentos/doc_002.jpg,impreso,,2 El secreto bancario y la reserva tributaria ...,PaddleOCR 3.7.0 (lang=es),,120.5
2,doc_003,datos/raw/documentos/doc_003.jpg,impreso,,PUNTO CENTRO DE UNIVERSIDAD INVESTIGACIÓN DEL ...,PaddleOCR 3.7.0 (lang=es),,104.8
3,doc_004,datos/raw/documentos/doc_004.jpg,impreso,,causales de juicio político (impeachment) cont...,PaddleOCR 3.7.0 (lang=es),,99.3
4,doc_005,datos/raw/documentos/doc_005.jpg,impreso,,"donde concentración, poder de mercado y eficie...",PaddleOCR 3.7.0 (lang=es),,124.4


In [3]:
import pandas as pd
d = pd.read_csv("datos/documentos.csv")
print(d.shape)
d["n_car"] = d["texto_ocr"].fillna("").str.len()
print(d.groupby("tipo")["n_car"].describe()[["count","min","mean","max"]])
print(d[d["n_car"] < 50][["id","tipo","n_car"]])

(25, 7)
            count     min         mean     max
tipo                                          
foto          8.0   490.0  1319.875000  3269.0
impreso       8.0  1874.0  3087.500000  6393.0
manuscrito    9.0   559.0  1718.888889  2548.0
Empty DataFrame
Columns: [id, tipo, n_car]
Index: []


In [4]:
import pandas as pd

d = pd.read_csv("datos/documentos.csv")

OEA = "https://www.oas.org/juridico/spanish/per_res17.pdf"
CIUP = "https://ciup.up.edu.pe/analisis/reformas-para-recuperar-institucionalidad-y-estabilidad/"
TESIS = "https://repositorio.up.edu.pe/backend/api/core/bitstreams/09a61c4e-748e-4242-9f61-f3400c64cf44/content"
TC = "https://img.lpderecho.pe/wp-content/uploads/2024/01/Exp.-0006-2009-PI-TC-LPDerecho.pdf"
MINAM = "https://www.minam.gob.pe/wp-content/uploads/2013/08/plana_2011_al_2021.pdf"

NORMA = "norma oficial (DL 822, art. 9)"
ABIERTO = "acceso abierto, sin licencia de reutilización visible (© Universidad del Pacífico); uso académico solo para OCR"


LIBROS = {
    "doc_009": "Los doce apóstoles de la economía peruana, Francisco Durand",
    "doc_010": "Conservadurismo, Robert Nisbet",
    "doc_011": "La ilegitimidad del poder político en el Perú, Jorge Aragón y Diego Sánchez",
    "doc_012": "Conflicto Armado Interno mediante el Pensamiento Clásico, Anónimo",
    "doc_013": "La ilegitimidad del poder político en el Perú, Jorge Aragón y Diego Sánchez",
    "doc_014": "La ilegitimidad del poder político en el Perú, Jorge Aragón y Diego Sánchez",
    "doc_015": "El cuento de la criada, Margaret Atwood",
    "doc_016": "La ilegitimidad del poder político en el Perú, Jorge Aragón y Diego Sánchez",
}

info = {
    "doc_001": (OEA, NORMA, "URL pública"),
    "doc_002": (OEA, NORMA, "URL pública"),
    "doc_003": (CIUP, ABIERTO, "URL pública; imagen no publicada"),
    "doc_004": (CIUP, ABIERTO, "URL pública; imagen no publicada"),
    "doc_005": (TESIS, "acceso abierto en repositorio UP; licencia específica no verificada; uso académico solo para OCR", "URL pública; imagen no publicada"),
    "doc_006": (TC, NORMA, "URL pública (copia LP Derecho)"),
    "doc_007": (TC, NORMA, "URL pública (copia LP Derecho)"),
    "doc_008": (MINAM, NORMA + " (plan aprobado por decreto supremo)", "URL pública"),
}

filas = []
for i in d["id"]:
    n = int(i[4:])
    if i in info:
        url, lic, acc = info[i]
    elif n <= 16:
        libro = LIBROS.get(i, "libro por completar")
        url = f"foto propia de página de libro: {libro}"
        lic = "foto propia; texto con derechos de autor del libro, uso académico solo para OCR (no se publica imagen ni texto completo)"
        acc = "archivo propio, no público"
    else:
        url, lic, acc = "captura propia", "captura propia, texto ficticio", "archivo propio, no público"
    filas.append({"id": i, "modalidad": "documento", "fuente_url": url,
                  "licencia": lic, "acceso": acc})

f = pd.DataFrame(filas)
f.to_csv("datos/fuentes.csv", index=False)

d["fuente"] = [r.fuente_url + " | " + r.licencia + " | extraído 25/10/2026" for r in f.itertuples()]
d.to_csv("datos/documentos.csv", index=False)

print(f.shape)
f.head(10)

(25, 5)


,id,modalidad,fuente_url,licencia,acceso
0,doc_001,documento,https://www.oas.org/juridico/spanish/per_res17...,"norma oficial (DL 822, art. 9)",URL pública
1,doc_002,documento,https://www.oas.org/juridico/spanish/per_res17...,"norma oficial (DL 822, art. 9)",URL pública
2,doc_003,documento,https://ciup.up.edu.pe/analisis/reformas-para-...,"acceso abierto, sin licencia de reutilización ...",URL pública; imagen no publicada
3,doc_004,documento,https://ciup.up.edu.pe/analisis/reformas-para-...,"acceso abierto, sin licencia de reutilización ...",URL pública; imagen no publicada
4,doc_005,documento,https://repositorio.up.edu.pe/backend/api/core...,acceso abierto en repositorio UP; licencia esp...,URL pública; imagen no publicada
5,doc_006,documento,https://img.lpderecho.pe/wp-content/uploads/20...,"norma oficial (DL 822, art. 9)",URL pública (copia LP Derecho)
6,doc_007,documento,https://img.lpderecho.pe/wp-content/uploads/20...,"norma oficial (DL 822, art. 9)",URL pública (copia LP Derecho)
7,doc_008,documento,https://www.minam.gob.pe/wp-content/uploads/20...,"norma oficial (DL 822, art. 9) (plan aprobado ...",URL pública
8,doc_009,documento,foto propia de página de libro: Los doce apóst...,foto propia; texto con derechos de autor del l...,"archivo propio, no público"
9,doc_010,documento,foto propia de página de libro: Conservadurism...,foto propia; texto con derechos de autor del l...,"archivo propio, no público"


In [6]:
import re, pandas as pd
from collections import Counter

# 1) Cargar y guardar respaldo sin seudonimizar (queda fuera de Git: datos/raw/)
d = pd.read_csv("datos/documentos.csv")
d["texto_ocr"] = d["texto_ocr"].fillna("")
d.to_csv("datos/raw/documentos/ocr_sin_seudonimizar.csv", index=False)

# 2) Patrones automáticos
patrones = {
    "[DNI]": r"\b\d{8}\b",
    "[TELEFONO]": r"\b9\d{2}[\s-]?\d{3}[\s-]?\d{3}\b",
    "[CORREO]": r"[\w.\-]+@[\w.\-]+\.\w+",
}

# 3) Nombres (de más largo a más corto, para que no queden restos)
NOMBRES = [
    "Javier Ignacio Albán González", "Albán González", "Javier Albán",
    "Francisco Durand", "Durand",
    "Robert Nisbet", "Conersmo Éobieisbet", "Nisbet",
    "Jorge Aragón", "Jorge Araginy", "Diego Sánchez", "Aragón",
    "Margaret Atwood", "Atwood",
    "Ignacio Távara Pasapera", "Távara Pasapera", "Távara",
    "Rosa Quispe Mamani", "Hugo Salinas Paredes",
    "Pedro Huamán Ccori", "Lucía Torres Vega", "Julio Ramos Cárdenas",
    "Carmen Ruiz Salazar",
]

conteo = Counter()

def limpiar(t):
    for etq, p in patrones.items():
        t, n = re.subn(p, etq, t)
        conteo[etq] += n
    for k, nom in enumerate(NOMBRES):
        t, n = re.subn(re.escape(nom), f"[PERSONA_{k+1}]", t, flags=re.IGNORECASE)
        conteo["PERSONA"] += n
    return t

d["texto_ocr"] = d["texto_ocr"].apply(limpiar)

# 4) Reporte
print("Entidades reemplazadas:", dict(conteo))
print("Total:", sum(conteo.values()))

# 5) Candidatos que aún podrían ser nombres (revisar a ojo)
cand = Counter()
for t in d["texto_ocr"]:
    for m in re.findall(r"\b[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+ (?:[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+ ?){1,3}", t):
        cand[m.strip()] += 1
print(cand.most_common(40))

# 6) Vista previa de los textos con más riesgo (sentencia TC y tesis)
for i in ["doc_006", "doc_007", "doc_005"]:
    t = d.loc[d["id"] == i, "texto_ocr"].iloc[0]
    print("\n", i, "->", t[:1200])

# 7) Guardar el CSV seudonimizado
d.to_csv("datos/documentos.csv", index=False)
print("\nGuardado:", d.shape)

Entidades reemplazadas: {'[DNI]': 0, '[TELEFONO]': 0, '[CORREO]': 1, 'PERSONA': 4}
Total: 5
[('Poder Judicial', 9), ('Cambio Climático', 7), ('El Estado', 4), ('Tribunal Constitucional', 3), ('Open Banking', 3), ('Carrera Judicial', 3), ('Programa Nacional', 3), ('Open Finance', 2), ('Corte Suprema', 2), ('El Programa Nacional', 2), ('Costa Norte', 2), ('El Perú', 2), ('Bosques Secos', 2), ('Inversión Pública', 2), ('Código Penal', 1), ('Autor Reformas', 1), ('Banco Central', 1), ('Pagos Minoristas', 1), ('Sala Constitucional', 1), ('Congreso Constituyente Democrático', 1), ('Poder Constituyente', 1), ('Reforma Constitucional', 1), ('Código Procesal Constitucional', 1), ('Este Colegiado', 1), ('Procurador Público', 1), ('Audiencia Pública', 1), ('Plan Nacional', 1), ('Reforma Integral', 1), ('Código Civil', 1), ('Texto Único Ordenado', 1), ('Ley Orgánica', 1), ('Los Jueces', 1), ('Reservas Territoriales', 1), ('Comunidades Nativas', 1), ('Producción Permanente', 1), ('San Martín', 1), 

## Seudonimización
Entidades reemplazadas: DNI 0, teléfonos 0, correos 1, personas 4 (total 5).
Respaldo sin seudonimizar en `datos/raw/` (fuera de Git). Se realizó un revisión manual parcial de doc_005, doc_006 y doc_007.